In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("/content/train.csv")
later = pd.read_csv("/content/later.csv")
holdout = pd.read_csv("/content/holdout_features.csv")

datasets = {
    "TRAIN": train,
    "LATER": later,
    "HOLDOUT": holdout
}

for name, df in datasets.items():
    print(f"\n{'='*50}")
    print(name)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())


TRAIN
Shape: (1000, 12)

Columns:
['record_id', 'date', 'highway_distance_km', 'nearby_pumps', 'price_rs', 'trucks_passing', 'temp_c', 'staff', 'ownership', 'zone', 'litres_from_tank_dip_next_day', 'litres_per_day']

Data types:
record_id                         object
date                              object
highway_distance_km              float64
nearby_pumps                       int64
price_rs                         float64
trucks_passing                     int64
temp_c                           float64
staff                              int64
ownership                         object
zone                              object
litres_from_tank_dip_next_day      int64
litres_per_day                     int64
dtype: object

Missing values:
record_id                        0
date                             0
highway_distance_km              0
nearby_pumps                     0
price_rs                         0
trucks_passing                   0
temp_c                           0
st

In [3]:
print("TRAIN columns:")
print(train.columns.tolist())

print("\nHOLDOUT columns:")
print(holdout.columns.tolist())

print("\nCorrelation with target:")
print(train.select_dtypes("number").corr()["litres_per_day"])

TRAIN columns:
['record_id', 'date', 'highway_distance_km', 'nearby_pumps', 'price_rs', 'trucks_passing', 'temp_c', 'staff', 'ownership', 'zone', 'litres_from_tank_dip_next_day', 'litres_per_day']

HOLDOUT columns:
['record_id', 'date', 'highway_distance_km', 'nearby_pumps', 'price_rs', 'trucks_passing', 'temp_c', 'staff', 'ownership', 'zone']

Correlation with target:
highway_distance_km             -0.008099
nearby_pumps                    -0.236363
price_rs                        -0.089257
trucks_passing                   0.067150
temp_c                           0.066292
staff                            0.070584
litres_from_tank_dip_next_day    0.307786
litres_per_day                   1.000000
Name: litres_per_day, dtype: float64


In [4]:
numeric = train.select_dtypes(include="number").columns

for col in numeric:
    print(f"\n{col}")
    print(train.groupby("zone")[col].agg(
        ["min", "mean", "median", "max"]
    ))


highway_distance_km
             min      mean  median    max
zone                                     
City        0.17  7.615333   7.640  14.98
Highway     0.03  7.300000   7.110  14.98
Industrial  0.00  7.383150   7.365  14.96

nearby_pumps
            min      mean  median  max
zone                                  
City          0  5.158974     5.0   10
Highway       0  5.273504     5.0   10
Industrial    0  5.028634     5.0   10

price_rs
              min       mean   median     max
zone                                         
City        90.03  99.890410  100.080  109.98
Highway     90.00  99.727578   99.860  109.95
Industrial  90.00  99.798172   99.435  109.99

trucks_passing
            min         mean  median   max
zone                                      
City         11  1056.297436  1069.0  1988
Highway       0  1002.284900   995.0  1996
Industrial    0   984.566079   975.5  1995

temp_c
            min       mean  median   max
zone                                    

In [5]:
for col in train.select_dtypes("number"):
    print(f"\n{col}")
    print(train[col].value_counts().head(15))


highway_distance_km
highway_distance_km
9.35     5
12.95    5
2.31     4
7.26     4
0.68     4
13.24    4
13.70    4
7.27     4
12.17    4
2.81     4
6.64     4
3.31     3
11.01    3
3.95     3
7.01     3
Name: count, dtype: int64

nearby_pumps
nearby_pumps
4     125
9     115
5     112
7     109
2     102
8      94
6      90
1      83
3      81
10     46
0      43
Name: count, dtype: int64

price_rs
price_rs
93.01     4
100.26    4
104.38    4
105.76    4
103.55    4
106.29    3
99.92     3
90.76     3
99.33     3
109.73    3
107.80    3
99.41     3
98.40     3
92.34     3
92.93     3
Name: count, dtype: int64

trucks_passing
trucks_passing
850     5
1866    5
817     4
1360    4
58      4
791     3
169     3
1342    3
139     3
607     3
1688    3
647     3
1069    3
282     3
847     3
Name: count, dtype: int64

temp_c
temp_c
26.9    9
24.1    8
31.8    7
23.4    7
26.0    6
15.2    6
6.6     6
35.9    6
34.3    6
6.4     6
30.2    6
30.1    6
35.4    6
38.4    6
22.2    6
Name: co

In [6]:
print("Duplicate rows:", train.duplicated().sum())

print(train[train.duplicated(keep=False)])

Duplicate rows: 0
Empty DataFrame
Columns: [record_id, date, highway_distance_km, nearby_pumps, price_rs, trucks_passing, temp_c, staff, ownership, zone, litres_from_tank_dip_next_day, litres_per_day]
Index: []


In [10]:
df = train.sort_values("date").copy()

for col in df.select_dtypes("number"):
    groups = df[col].ne(df[col].shift()).cumsum()

    runs = df.groupby(groups).agg(
        value=(col, "first"),
        count=(col, "size"),
        start=("date", "first"),
        end=("date", "last")
    )

    suspicious = runs[runs["count"] >= 10]

    if not suspicious.empty:
        print(f"\nColumn: {col}")
        print(suspicious)

In [8]:
print(
    train["litres_per_day"]
    .sort_values(ascending=False)
    .to_string(index=False)
)

197660
173760
166470
152770
146270
140290
127310
114480
101140
 97050
 63430
 24785
 24535
 24523
 23856
 22742
 22712
 22696
 22357
 22018
 22009
 21911
 21904
 21780
 21603
 21592
 21591
 21533
 21530
 21453
 21431
 21397
 21297
 21129
 21000
 20935
 20908
 20896
 20871
 20867
 20857
 20826
 20804
 20781
 20722
 20648
 20585
 20556
 20459
 20266
 20188
 20078
 20066
 20037
 20002
 19938
 19872
 19842
 19774
 19756
 19728
 19706
 19667
 19613
 19578
 19569
 19551
 19452
 19422
 19420
 19400
 19369
 19369
 19314
 19281
 19278
 19275
 19261
 19253
 19253
 19235
 19227
 19222
 19199
 19174
 19165
 19155
 19134
 19123
 19104
 19063
 19044
 19042
 19015
 19014
 19010
 19003
 18985
 18984
 18926
 18887
 18876
 18828
 18821
 18819
 18805
 18790
 18782
 18662
 18644
 18628
 18627
 18626
 18609
 18582
 18580
 18530
 18484
 18466
 18451
 18433
 18424
 18407
 18397
 18387
 18359
 18358
 18354
 18350
 18343
 18341
 18311
 18239
 18180
 18177
 18164
 18162
 18142
 18109
 18098
 18084
 18022
 18008

In [9]:
for col in train.select_dtypes("object"):
    print(f"\nTRAIN: {col}")
    print(train[col].value_counts().to_string())

    if col in later.columns:
        print(f"\nLATER: {col}")
        print(later[col].value_counts().to_string())


TRAIN: record_id
record_id
R00001    1
R00002    1
R00003    1
R00004    1
R00005    1
R00006    1
R00007    1
R00008    1
R00009    1
R00010    1
R00011    1
R00012    1
R00013    1
R00014    1
R00015    1
R00016    1
R00017    1
R00018    1
R00019    1
R00020    1
R00021    1
R00022    1
R00023    1
R00024    1
R00025    1
R00026    1
R00027    1
R00028    1
R00029    1
R00030    1
R00031    1
R00032    1
R00033    1
R00034    1
R00035    1
R00036    1
R00037    1
R00038    1
R00039    1
R00040    1
R00041    1
R00042    1
R00043    1
R00044    1
R00045    1
R00046    1
R00047    1
R00048    1
R00049    1
R00050    1
R00051    1
R00052    1
R00053    1
R00054    1
R00055    1
R00056    1
R00057    1
R00058    1
R00059    1
R00060    1
R00061    1
R00062    1
R00063    1
R00064    1
R00065    1
R00066    1
R00067    1
R00068    1
R00069    1
R00070    1
R00071    1
R00072    1
R00073    1
R00074    1
R00075    1
R00076    1
R00077    1
R00078    1
R00079    1
R00080    1
R00081    1
